# Optional brain-extraction controls for the same 10 mouse T1 images

This companion notebook leaves the working MouseBrainExtractor/RS2-Net benchmark unchanged. It adds two deliberately labelled **controls**, not two equally validated mouse-T1 candidates:

1. **CAMRI rodent 2D U-Net** — open rodent weights, but trained on T2-weighted RARE and T2*-weighted EPI rather than T1.
2. **deepbet** — a modern T1 model, but trained for healthy adult human brains rather than mice.

A good-looking result is useful evidence for this cohort, but it does not erase either training-domain mismatch. Keep MBE `invivo_iso`, MBE `invivo_aniso`, and RS2-Net as the primary comparison.

## Exact use

1. In Colab choose **Runtime -> Change runtime type -> T4 GPU**.
2. Choose **Runtime -> Run all**.
3. Upload the same `t1_brain_extraction_benchmark_10.zip` used by the primary notebook.
4. Download `t1_brain_extraction_extra_results.zip` from the final cell.
5. On your Mac, pass both downloaded result archives to the repository's ITK-SNAP review command.

All outputs are automatic pre-labels and require human review.


In [ ]:
# Configuration. Set either switch to False if you want only one extra control.
from pathlib import Path
import os, shutil, subprocess, sys
os.environ['KERAS_BACKEND'] = 'torch'  # Must be set before importing Keras.
import torch

RUN_CAMRI_RODENT_UNET = True
RUN_DEEPBET = True
CAMRI_BATCH_SIZE = 16  # Conservative T4 setting for 128 x 128 patches.
CAMRI_COMMIT = 'dfa1a123495628d5d8ffe576999f8f1ddfd1973a'
CAMRI_REPOSITORY = 'https://github.com/CAMRIatUNC/RodentMRISkullStripping.git'
DEEPBET_REPOSITORY = 'https://github.com/wwu-mmll/deepbet'
DEEPBET_VERSION = '1.0.2'

BASE = Path('/content/lys_brain_extra_benchmark')
PACKAGE_AREA = BASE / 'package'
EXTERNAL = BASE / 'external'
WORK = BASE / 'work'
RESULTS = BASE / 't1_brain_extraction_extra_results'
for directory in (PACKAGE_AREA, EXTERNAL, WORK, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Select Runtime -> Change runtime type -> T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)


In [ ]:
# Upload and unpack the same frozen 10-image package used by the primary notebook.
from google.colab import files
import csv, json, zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Upload exactly one .zip package; received: {list(uploaded)}')
upload_path = Path('/content') / zip_names[0]
if PACKAGE_AREA.exists():
    shutil.rmtree(PACKAGE_AREA)
PACKAGE_AREA.mkdir(parents=True)
with zipfile.ZipFile(upload_path) as archive:
    for member in archive.infolist():
        target = (PACKAGE_AREA / member.filename).resolve()
        if not target.is_relative_to(PACKAGE_AREA.resolve()):
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(PACKAGE_AREA)
manifests = list(PACKAGE_AREA.rglob('benchmark_manifest.csv'))
if len(manifests) != 1:
    raise ValueError(f'Expected one benchmark_manifest.csv, found {len(manifests)}')
PACKAGE_ROOT = manifests[0].parent
with manifests[0].open(newline='') as stream:
    PACKAGE_ROWS = list(csv.DictReader(stream))
if len(PACKAGE_ROWS) != 10:
    raise ValueError(f'This benchmark expects exactly 10 cases, found {len(PACKAGE_ROWS)}')
if RESULTS.exists():
    shutil.rmtree(RESULTS)
(RESULTS / 'inputs').mkdir(parents=True)
CASES = []
for row in PACKAGE_ROWS:
    source = PACKAGE_ROOT / row['image']
    destination = RESULTS / 'inputs' / f"{row['case_id']}_pre_t1.nii.gz"
    shutil.copy2(source, destination)
    CASES.append({'case_id': row['case_id'], 'image': destination})
shutil.copy2(manifests[0], RESULTS / 'input_benchmark_manifest.csv')
if (PACKAGE_ROOT / 'package_metadata.json').is_file():
    shutil.copy2(PACKAGE_ROOT / 'package_metadata.json', RESULTS / 'input_package_metadata.json')
print('Cases:', ', '.join(row['case_id'] for row in CASES))


In [ ]:
# Install released packages, clone the pinned CAMRI source, and fail early if APIs drift.
packages = [
    'deepbet==1.0.2', 'keras==3.15.0', 'h5py>=3.14,<4',
    'connected-components-3d>=3.20,<5', 'fill-voids>=2.0,<3',
    'nibabel>=5.3,<6', 'SimpleITK>=2.4,<3', 'pandas', 'tqdm', 'matplotlib'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

def clone_pinned(repository, commit, destination):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.check_call(['git', 'clone', '-q', repository, str(destination)])
    subprocess.check_call(['git', '-C', str(destination), 'checkout', '-q', commit])
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != commit:
        raise RuntimeError(f'Pinned checkout failed: expected {commit}, got {actual}')

CAMRI_ROOT = EXTERNAL / 'RodentMRISkullStripping'
clone_pinned(CAMRI_REPOSITORY, CAMRI_COMMIT, CAMRI_ROOT)
CAMRI_WEIGHT = CAMRI_ROOT / 'rbm' / 'scripts' / 'rat_brain-2d_unet.hdf5'
if not CAMRI_WEIGHT.is_file():
    raise FileNotFoundError(f'CAMRI checkpoint missing: {CAMRI_WEIGHT}')

import numpy as np
import keras
from keras.models import load_model
from deepbet.bet import BrainExtraction
from deepbet.utils import DATA_PATH as DEEPBET_DATA_PATH
if keras.backend.backend() != 'torch':
    raise RuntimeError(f'Expected Keras torch backend, got {keras.backend.backend()}')
CAMRI_MODEL = load_model(CAMRI_WEIGHT, compile=False)
probe = CAMRI_MODEL.predict(np.zeros((1, 128, 128, 1), dtype=np.float32), verbose=0)
if isinstance(probe, list):
    probe = probe[0]
if tuple(probe.shape) != (1, 128, 128, 1):
    raise RuntimeError(f'Unexpected CAMRI output shape: {probe.shape}')
DEEPBET_MODEL = BrainExtraction(no_gpu=False)
print('Runtime preflight: Keras', keras.__version__, 'backend', keras.backend.backend())
print('Model preflight: CAMRI checkpoint and deepbet models loaded successfully')


In [ ]:
# Shared output contract, logging, provenance, and NIfTI standardization.
import contextlib, hashlib, importlib.metadata, importlib.util, platform, time
from datetime import datetime, timezone
import nibabel as nib
from nibabel.processing import resample_from_to

RUN_FIELDS = ['case_id', 'model_id', 'status', 'image', 'mask', 'metadata', 'log', 'message']
RUN_ROWS = {}

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def relative(path):
    return str(Path(path).resolve().relative_to(RESULTS.resolve())) if path else ''

def write_run_manifest():
    with (RESULTS / 'run_manifest.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=RUN_FIELDS)
        writer.writeheader()
        writer.writerows(RUN_ROWS[key] for key in sorted(RUN_ROWS))

def record(case_id, model_id, status, image, mask=None, metadata=None, log=None, message=''):
    RUN_ROWS[(case_id, model_id)] = {
        'case_id': case_id, 'model_id': model_id, 'status': status,
        'image': relative(image), 'mask': relative(mask),
        'metadata': relative(metadata), 'log': relative(log), 'message': message,
    }
    write_run_manifest()

def standardize_mask(source, reference, destination):
    reference_image = nib.load(str(reference))
    mask_image = nib.load(str(source))
    resampled = False
    if mask_image.shape != reference_image.shape or not np.allclose(mask_image.affine, reference_image.affine, rtol=1e-5, atol=1e-5):
        mask_image = resample_from_to(mask_image, reference_image, order=0)
        resampled = True
    mask = (np.asanyarray(mask_image.dataobj) > 0).astype(np.uint8)
    if not mask.any() or mask.all():
        raise ValueError(f'Invalid constant mask after standardization: foreground={int(mask.sum())}')
    header = reference_image.header.copy()
    header.set_data_dtype(np.uint8)
    destination.parent.mkdir(parents=True, exist_ok=True)
    output = nib.Nifti1Image(mask, reference_image.affine, header)
    output.set_qform(reference_image.affine, code=int(reference_image.header['qform_code']))
    output.set_sform(reference_image.affine, code=int(reference_image.header['sform_code']))
    nib.save(output, destination)
    return {'resampled_to_input_grid': resampled, 'foreground_voxels': int(mask.sum())}

deepbet_weights = sorted(Path(DEEPBET_DATA_PATH).rglob('*.pt'))
if len(deepbet_weights) < 2:
    raise FileNotFoundError(f'Expected bundled deepbet weights, found {deepbet_weights}')
RUNTIME = {
    'created_at': utc_now(), 'python': platform.python_version(),
    'pytorch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0), 'keras': keras.__version__,
    'keras_backend': keras.backend.backend(),
}
MODEL_PROVENANCE = {
    'camri_rodent_unet_t2': {
        'repository': CAMRI_REPOSITORY, 'commit': CAMRI_COMMIT,
        'weight_sha256': sha256(CAMRI_WEIGHT),
        'training_domain': 'rat and mouse T2-weighted RARE and T2*-weighted EPI',
        'domain_warning': 'Cross-contrast control: the released model was not trained for T1-weighted MRI.',
    },
    'deepbet_human_t1': {
        'repository': DEEPBET_REPOSITORY, 'package_version': DEEPBET_VERSION,
        'weight_sha256': {path.name: sha256(path) for path in deepbet_weights},
        'training_domain': 'healthy adult human T1-weighted MRI',
        'domain_warning': 'Cross-species control: the released model was not trained for mouse MRI.',
    },
}
print(json.dumps({'runtime': RUNTIME, 'models': MODEL_PROVENANCE}, indent=2))


In [ ]:
# Human-T1 cross-species control using deepbet's released inference and postprocessing.
model_id = 'deepbet_human_t1'
if RUN_DEEPBET:
    for index, case in enumerate(CASES, start=1):
        case_id, image = case['case_id'], case['image']
        print(f'[{index}/{len(CASES)}] {model_id}: {case_id}')
        raw = WORK / model_id / case_id / f'{case_id}_raw.nii.gz'
        mask = RESULTS / 'predictions' / model_id / f'{case_id}_brain_mask.nii.gz'
        metadata = RESULTS / 'metadata' / model_id / f'{case_id}.json'
        log = RESULTS / 'logs' / model_id / f'{case_id}.log'
        for path in (raw.parent, metadata.parent, log.parent):
            path.mkdir(parents=True, exist_ok=True)
        started = time.monotonic()
        try:
            with log.open('w') as stream, contextlib.redirect_stdout(stream), contextlib.redirect_stderr(stream):
                DEEPBET_MODEL.run(str(image), mask_path=str(raw), threshold=0.5, n_dilate=0)
            standard = standardize_mask(raw, image, mask)
            payload = {
                'case_id': case_id, 'model_id': model_id, 'status': 'ok',
                'input': relative(image), 'output': relative(mask),
                'preprocessing': 'official deepbet canonical reorientation, bounding-box localization, resizing, and quantile normalization',
                'official_postprocessing': 'threshold 0.5, largest connected component, and hole filling; no dilation',
                'runtime_seconds': round(time.monotonic() - started, 3),
                'input_sha256': sha256(image), 'mask_sha256': sha256(mask),
                **standard, **MODEL_PROVENANCE[model_id], **RUNTIME,
            }
            metadata.write_text(json.dumps(payload, indent=2) + '\n')
            record(case_id, model_id, 'ok', image, mask, metadata, log)
        except Exception as exc:
            message = f'{type(exc).__name__}: {exc}'
            with log.open('a') as stream:
                stream.write('\nBENCHMARK ERROR: ' + message + '\n')
            record(case_id, model_id, 'failed', image, log=log, message=message)
            print('FAILED:', message)
        finally:
            torch.cuda.empty_cache()
else:
    print('Skipped deepbet_human_t1 by configuration.')


In [ ]:
# Rodent cross-contrast control. This is a batch-equivalent implementation of
# the official 0.1-mm resampling, min-max normalization, bidirectional patch
# traversal, 0.5 threshold, categorical voting, and nearest-neighbour return.
import SimpleITK as sitk

def resample_sitk(image, spacing, interpolator, size=None):
    spacing = tuple(float(value) for value in spacing)
    if size is None:
        size = np.ceil(np.asarray(image.GetSize()) * (np.asarray(image.GetSpacing()) / np.asarray(spacing))).astype(int)
    operation = sitk.ResampleImageFilter()
    operation.SetInterpolator(interpolator)
    operation.SetOutputDirection(image.GetDirection())
    operation.SetOutputOrigin(image.GetOrigin())
    operation.SetOutputSpacing(spacing)
    operation.SetSize([int(value) for value in size])
    return operation.Execute(image)

def camri_descriptors(shape, patch=128, stride=32):
    length, columns, rows = shape
    if columns < patch or rows < patch:
        raise ValueError(f'CAMRI resampled in-plane shape is too small for 128 patches: {shape}')
    descriptors = []
    for index in range(0, length):
        for column in range(0, columns - patch + 1, stride):
            for row in range(0, rows - patch + 1, stride):
                descriptors.append((index, column, row))
    for end_index in range(length, 0, -1):
        for end_column in range(columns, patch - 1, -stride):
            for end_row in range(rows, patch - 1, -stride):
                descriptors.append((end_index - 1, end_column - patch, end_row - patch))
    return descriptors

def camri_predict(input_path, output_path):
    original = sitk.ReadImage(str(input_path))
    resampled = resample_sitk(original, (0.1, 0.1, 0.1), sitk.sitkLinear)
    volume = sitk.GetArrayFromImage(resampled).astype(np.float32)
    minimum, maximum = float(np.min(volume)), float(np.max(volume))
    if not np.isfinite(minimum) or not np.isfinite(maximum) or maximum <= minimum:
        raise ValueError(f'Cannot min-max normalize input range {minimum}..{maximum}')
    volume = (volume - minimum) / (maximum - minimum)
    descriptors = camri_descriptors(volume.shape)
    foreground_votes = np.zeros(volume.shape, dtype=np.uint16)
    total_votes = np.zeros(volume.shape, dtype=np.uint16)
    for start in range(0, len(descriptors), CAMRI_BATCH_SIZE):
        batch_descriptors = descriptors[start:start + CAMRI_BATCH_SIZE]
        patches = np.stack([volume[z, y:y + 128, x:x + 128] for z, y, x in batch_descriptors])[..., None]
        probabilities = CAMRI_MODEL.predict(patches, batch_size=len(batch_descriptors), verbose=0)
        if isinstance(probabilities, list):
            probabilities = probabilities[0]
        probabilities = np.asarray(probabilities)[..., 0]
        for (z, y, x), probability in zip(batch_descriptors, probabilities):
            foreground_votes[z, y:y + 128, x:x + 128] += (probability >= 0.5)
            total_votes[z, y:y + 128, x:x + 128] += 1
    label = (foreground_votes > (total_votes / 2.0)).astype(np.uint8)
    label_image = sitk.GetImageFromArray(label)
    label_image.CopyInformation(resampled)
    restored = resample_sitk(label_image, original.GetSpacing(), sitk.sitkNearestNeighbor, original.GetSize())
    sitk.WriteImage(restored, str(output_path))
    return {
        'official_resampled_shape_zyx': list(volume.shape),
        'patch_count': len(descriptors),
        'covered_resampled_voxel_fraction': float(np.mean(total_votes > 0)),
        'input_intensity_min': minimum, 'input_intensity_max': maximum,
    }

model_id = 'camri_rodent_unet_t2'
if RUN_CAMRI_RODENT_UNET:
    for index, case in enumerate(CASES, start=1):
        case_id, image = case['case_id'], case['image']
        print(f'[{index}/{len(CASES)}] {model_id}: {case_id}')
        raw = WORK / model_id / case_id / f'{case_id}_raw.nii.gz'
        mask = RESULTS / 'predictions' / model_id / f'{case_id}_brain_mask.nii.gz'
        metadata = RESULTS / 'metadata' / model_id / f'{case_id}.json'
        log = RESULTS / 'logs' / model_id / f'{case_id}.log'
        for path in (raw.parent, metadata.parent, log.parent):
            path.mkdir(parents=True, exist_ok=True)
        started = time.monotonic()
        try:
            with log.open('w') as stream, contextlib.redirect_stdout(stream), contextlib.redirect_stderr(stream):
                adapter = camri_predict(image, raw)
            standard = standardize_mask(raw, image, mask)
            payload = {
                'case_id': case_id, 'model_id': model_id, 'status': 'ok',
                'input': relative(image), 'output': relative(mask),
                'preprocessing': 'official 0.1-mm isotropic linear resampling and global min-max normalization',
                'inference': 'batch-equivalent official 128x128 bidirectional patch traversal, stride 32, threshold 0.5, categorical majority voting',
                'postprocessing': 'official nearest-neighbour resampling to the native input grid; no added morphology',
                'runtime_seconds': round(time.monotonic() - started, 3),
                'input_sha256': sha256(image), 'mask_sha256': sha256(mask),
                **adapter, **standard, **MODEL_PROVENANCE[model_id], **RUNTIME,
            }
            metadata.write_text(json.dumps(payload, indent=2) + '\n')
            record(case_id, model_id, 'ok', image, mask, metadata, log)
        except Exception as exc:
            message = f'{type(exc).__name__}: {exc}'
            with log.open('a') as stream:
                stream.write('\nBENCHMARK ERROR: ' + message + '\n')
            record(case_id, model_id, 'failed', image, log=log, message=message)
            print('FAILED:', message)
        finally:
            torch.cuda.empty_cache()
else:
    print('Skipped camri_rodent_unet_t2 by configuration.')


In [ ]:
# Create one immediate visual comparison montage per case.
import matplotlib.pyplot as plt
from IPython.display import display, Image

MODEL_ORDER = ['camri_rodent_unet_t2', 'deepbet_human_t1']
MODEL_TITLES = {
    'camri_rodent_unet_t2': 'CAMRI rodent T2 control',
    'deepbet_human_t1': 'deepbet human T1 control',
}
qc_dir = RESULTS / 'qc'
qc_dir.mkdir(parents=True, exist_ok=True)
created_qc = []
for case in CASES:
    case_id, image_path = case['case_id'], case['image']
    available = [model for model in MODEL_ORDER if RUN_ROWS.get((case_id, model), {}).get('status') == 'ok']
    if not available:
        continue
    image_data = nib.load(str(image_path)).get_fdata(dtype=np.float32)
    nonzero = image_data[np.isfinite(image_data) & (image_data != 0)]
    vmin, vmax = np.percentile(nonzero, [1, 99.5]) if nonzero.size else (0, 1)
    start, stop = min(50, image_data.shape[2] - 1), min(170, image_data.shape[2] - 1)
    slices = np.linspace(start, stop, 6).round().astype(int)
    fig, axes = plt.subplots(len(available), len(slices), figsize=(15, 2.7 * len(available)), squeeze=False)
    for row_index, current_model in enumerate(available):
        mask_path = RESULTS / RUN_ROWS[(case_id, current_model)]['mask']
        mask_data = np.asanyarray(nib.load(str(mask_path)).dataobj) > 0
        for column_index, slice_index in enumerate(slices):
            axis = axes[row_index, column_index]
            axis.imshow(np.rot90(image_data[:, :, slice_index]), cmap='gray', vmin=vmin, vmax=vmax)
            axis.contour(np.rot90(mask_data[:, :, slice_index]), levels=[0.5], colors='lime', linewidths=0.8)
            axis.set_xticks([]); axis.set_yticks([])
            if row_index == 0:
                axis.set_title(f'slice {slice_index}', fontsize=9)
            if column_index == 0:
                axis.set_ylabel(MODEL_TITLES[current_model], fontsize=9)
    fig.suptitle(f'{case_id}: optional control boundaries', fontsize=12)
    fig.tight_layout()
    output = qc_dir / f'{case_id}_extra_model_comparison.png'
    fig.savefig(output, dpi=160, bbox_inches='tight')
    plt.close(fig)
    created_qc.append(output)
print(f'Created {len(created_qc)} QC montages.')
if created_qc:
    display(Image(filename=str(created_qc[0])))


In [ ]:
# Validate, record provenance, zip the standardized masks, and download them.
validation_rows = []
for key in sorted(RUN_ROWS):
    row = RUN_ROWS[key]
    validation = {'case_id': row['case_id'], 'model_id': row['model_id'], 'run_status': row['status'], 'grid_ok': '', 'binary_ok': '', 'message': row['message']}
    if row['status'] == 'ok':
        try:
            image = nib.load(str(RESULTS / row['image']))
            mask = nib.load(str(RESULTS / row['mask']))
            values = np.unique(np.asanyarray(mask.dataobj))
            validation['grid_ok'] = bool(image.shape == mask.shape and np.allclose(image.affine, mask.affine, rtol=1e-5, atol=1e-5))
            validation['binary_ok'] = bool(set(values.tolist()).issubset({0, 1}) and values.size > 1)
            if not validation['grid_ok'] or not validation['binary_ok']:
                validation['message'] = f'grid_ok={validation["grid_ok"]}, binary_ok={validation["binary_ok"]}, values={values[:8]}'
        except Exception as exc:
            validation['message'] = f'{type(exc).__name__}: {exc}'
    validation_rows.append(validation)
with (RESULTS / 'validation_summary.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['case_id', 'model_id', 'run_status', 'grid_ok', 'binary_ok', 'message'])
    writer.writeheader(); writer.writerows(validation_rows)

run_metadata = {
    'purpose': 'Optional diagnostic controls for the primary mouse T1 brain-extraction benchmark',
    'created_at': utc_now(), 'case_count': len(CASES),
    'case_ids': [case['case_id'] for case in CASES],
    'model_provenance': MODEL_PROVENANCE, 'runtime': RUNTIME,
    'warning': 'Both models have declared domain mismatches. Outputs are automatic pre-labels requiring review.',
}
(RESULTS / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2) + '\n')
write_run_manifest()
ok = sum(row['status'] == 'ok' for row in RUN_ROWS.values())
failed = sum(row['status'] != 'ok' for row in RUN_ROWS.values())
print(f'Predictions ready: {ok}; failed/skipped: {failed}; expected when all enabled: {len(CASES) * 2}')
for row in RUN_ROWS.values():
    if row['status'] != 'ok':
        print('FAILED:', row['case_id'], row['model_id'], row['message'])
archive_base = Path('/content/t1_brain_extraction_extra_results')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name))
print(f'Downloading {archive_path} ({archive_path.stat().st_size / (1024**2):.1f} MB)')
files.download(str(archive_path))
